# Section 5: Autoencoders and Latent-Space Thinking

**Student:** `<name>`  
**GitHub username:** `<username>`  
**Date:** `<date>`

This notebook is a field guide to three different latent spaces. Complete every `TODO`, keep the required fixed splits, and restart/run all cells before submission.

## 0. Setup and reproducibility

Run from the repository environment with `uv run jupyter lab`. CPU is sufficient. Fashion-MNIST downloads to `.course-data/` on first use and must not be committed.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
from torchvision import datasets, transforms

from sklearn.datasets import load_digits, load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

GLOBAL_SEED = 42
SPLIT_SEED = 42
NOISE_SEED = 31_415
DEVICE = torch.device("cpu")

def set_seed(seed: int = GLOBAL_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")

def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def assert_disjoint(*index_groups: np.ndarray) -> None:
    sets = [set(np.asarray(group).tolist()) for group in index_groups]
    for left in range(len(sets)):
        for right in range(left + 1, len(sets)):
            assert sets[left].isdisjoint(sets[right]), (left, right, sets[left] & sets[right])

set_seed()
REPOSITORY_ROOT = find_repository_root()
DATA_ROOT = REPOSITORY_ROOT / ".course-data"
print({
    "repository_root": str(REPOSITORY_ROOT),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "sklearn": sklearn.__version__,
})

## 1. Latent-space contracts

Complete this table **before training**.

| Lab | Input shape/range | Target shape/range | Code | Final activation | Loss and normalization | Constraint | Validation rule |
|---|---|---|---|---|---|---|---|
| Wine undercomplete AE | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| Digits plain/denoising AE | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| Fashion-MNIST VAE | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

For each lab, draw or write the full encoder–latent–decoder path with tensor shapes. Explain why the model cannot be evaluated from a colored latent scatter plot alone.

# Lab A — Dense Undercomplete Autoencoder on UCI Wine

The cultivar label is used to create the required fixed stratified split. It must not enter the scaler, PCA, autoencoder, reconstruction loss, checkpoint selection, or architecture choice.

In [ ]:
wine = load_wine()
X_wine = wine.data.astype(np.float32)
y_wine = wine.target.astype(np.int64)
wine_indices = np.arange(len(X_wine))

wine_train_val_idx, wine_test_idx = train_test_split(
    wine_indices, test_size=0.20, random_state=SPLIT_SEED, stratify=y_wine
)
wine_train_idx, wine_val_idx = train_test_split(
    wine_train_val_idx,
    test_size=0.25,
    random_state=SPLIT_SEED,
    stratify=y_wine[wine_train_val_idx],
)
assert_disjoint(wine_train_idx, wine_val_idx, wine_test_idx)

wine_scaler = StandardScaler().fit(X_wine[wine_train_idx])
X_wine_train = wine_scaler.transform(X_wine[wine_train_idx]).astype(np.float32)
X_wine_val = wine_scaler.transform(X_wine[wine_val_idx]).astype(np.float32)
X_wine_test = wine_scaler.transform(X_wine[wine_test_idx]).astype(np.float32)

wine_pca = PCA(n_components=2, random_state=GLOBAL_SEED).fit(X_wine_train)
Z_wine_pca_train = wine_pca.transform(X_wine_train)
Z_wine_pca_val = wine_pca.transform(X_wine_val)
Z_wine_pca_test = wine_pca.transform(X_wine_test)

wine_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_wine_train)),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
wine_val_loader = DataLoader(TensorDataset(torch.from_numpy(X_wine_val)), batch_size=128)
wine_test_loader = DataLoader(TensorDataset(torch.from_numpy(X_wine_test)), batch_size=128)

print("split sizes:", len(wine_train_idx), len(wine_val_idx), len(wine_test_idx))
print("feature names:", list(wine.feature_names))
print("training means after scaling:", np.round(X_wine_train.mean(axis=0), 5))
print("training std after scaling:", np.round(X_wine_train.std(axis=0), 5))

### A1. Audit the split and preprocessing

Report class counts by split, raw feature ranges, missing values, and the numerical checks proving the scaler and PCA were fit on training data only. Why must the Wine decoder output remain unrestricted?

In [ ]:
# TODO: complete the Wine split and preprocessing audit.

### A2. Define the two-dimensional autoencoder

Use a small dense network. Implement explicit `encode` and `decode` methods so that later latent-space tests do not depend on hooks or copied layers.

In [ ]:
class WineAutoencoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # TODO: define a small encoder ending in 2 values and a decoder ending in 13 unrestricted values.

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

# TODO: instantiate the model, print it, print its parameter count, and verify (batch, 13) -> (batch, 2) -> (batch, 13).

### A3. Train, validate, and lock the checkpoint

Record the plan before running.

| Architecture | Optimizer | Learning rate | Batch size | Max epochs | Stopping/checkpoint rule | Seed |
|---|---|---:|---:|---:|---|---:|
| TODO | TODO | TODO | TODO | TODO | TODO | 42 |

Use mean squared error in standardized feature space. Save a deep copy of the best validation checkpoint.

In [ ]:
# TODO: train the Wine AE, record training/validation loss per epoch, and restore the selected checkpoint.
# Save the history, best epoch, approximate training time, and model metadata.

In [ ]:
# TODO: plot Wine training and validation loss. Diagnose the curves.

**Locked Wine checkpoint and selection rule:** TODO

Do not change the model after the next cell evaluates the test set.

In [ ]:
# TODO: evaluate locked PCA and AE reconstructions on the Wine test set with the same MSE reduction.
# Show at least five rows and per-feature MSE in original units after inverse transformation.

In [ ]:
# TODO: plot PCA and AE coordinates for all splits using split markers.
# Make an unlabeled plot first. Only then color a second version by cultivar.

### A4. Frozen-representation probes

Use the same fixed `LogisticRegression(max_iter=5000, C=1.0, random_state=GLOBAL_SEED)` probe on raw standardized, two-dimensional PCA, and two-dimensional AE representations. Fit probes on the training split only. No hyperparameter search is required.

In [ ]:
# TODO: fit the three fixed probes, report held-out accuracy, and display all three confusion matrices.

In [ ]:
WINE_QUERY_TEST_POSITIONS = [0, 1]
WINE_INTERPOLATION_TEST_POSITIONS = [0, 1]

# TODO: for each fixed query, show its three nearest training codes and compare named original measurements.
# TODO: decode 11 evenly spaced points between the fixed endpoint codes, inverse-transform them,
# and plot at least four named measurements along the path.

### A5. Wine conclusions

1. Which evidence evaluates reconstruction, neighborhood structure, and downstream usefulness? **TODO**
2. Why can cultivar clusters appear without supervised AE training? **TODO**
3. Can lower reconstruction MSE and lower probe accuracy coexist? **TODO**
4. What does the interpolation show, and what claim would go too far? **TODO**

# Lab B — Convolutional Denoising Autoencoder on Digits

The plain and denoising models must use the same architecture and split. The only training-contract difference is clean input versus Gaussian-corrupted input; both target the clean image.

In [ ]:
digits = load_digits()
X_digits = (digits.images.astype(np.float32) / 16.0)[:, None, :, :]
y_digits = digits.target.astype(np.int64)
digit_indices = np.arange(len(X_digits))

digit_train_val_idx, digit_test_idx = train_test_split(
    digit_indices, test_size=0.20, random_state=SPLIT_SEED, stratify=y_digits
)
digit_train_idx, digit_val_idx = train_test_split(
    digit_train_val_idx,
    test_size=0.25,
    random_state=SPLIT_SEED,
    stratify=y_digits[digit_train_val_idx],
)
assert_disjoint(digit_train_idx, digit_val_idx, digit_test_idx)

X_digit_train = torch.from_numpy(X_digits[digit_train_idx])
X_digit_val = torch.from_numpy(X_digits[digit_val_idx])
X_digit_test = torch.from_numpy(X_digits[digit_test_idx])

digit_train_loader = DataLoader(
    TensorDataset(X_digit_train),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
digit_val_loader = DataLoader(TensorDataset(X_digit_val), batch_size=256)
digit_test_loader = DataLoader(TensorDataset(X_digit_test), batch_size=256)

def gaussian_corruption(x: torch.Tensor, sigma: float, seed: int) -> torch.Tensor:
    generator = torch.Generator(device=x.device).manual_seed(seed)
    noise = torch.randn(x.shape, dtype=x.dtype, device=x.device, generator=generator)
    return torch.clamp(x + sigma * noise, 0.0, 1.0)

DIGIT_NOISE_LEVELS = [0.0, 0.2, 0.35, 0.5]
X_digit_test_noisy = {
    sigma: gaussian_corruption(X_digit_test, sigma=sigma, seed=NOISE_SEED + i)
    for i, sigma in enumerate(DIGIT_NOISE_LEVELS)
}
DIGIT_DISPLAY_POSITIONS = np.arange(12)

print("split sizes:", len(digit_train_idx), len(digit_val_idx), len(digit_test_idx))
print("image shape/range:", X_digit_train.shape[1:], (X_digit_train.min().item(), X_digit_train.max().item()))

### B1. Define one convolutional architecture

Use the same class and initial seed for the plain and denoising runs. The encoder should reduce the 8-by-8 image to a small vector; the decoder must return `(N, 1, 8, 8)` in `[0, 1]`.

In [ ]:
class DigitConvAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 8) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: define convolutional encoder, latent layer, and decoder.

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

# TODO: instantiate one model and print actual shapes after every major stage for a real batch.
# Assert that the final shape equals the target shape and values are in the declared range.

### B2. Matched training plan

| Run | Training input -> target | Architecture | Optimizer | Learning rate | Batch size | Max epochs | Checkpoint rule |
|---|---|---|---|---:|---:|---:|---|
| Plain AE | clean -> clean | TODO | TODO | TODO | 64 | TODO | TODO |
| Denoising AE | Gaussian sigma 0.35 -> clean | same | same | same | 64 | same | same |

For the denoising training loop, generate reproducible new noise as training proceeds. Use fixed validation corruption so epochs are comparable.

In [ ]:
# TODO: train the matched plain and denoising AEs.
# Record train/validation per-pixel MSE, best epochs, metadata, and approximate training times.
# Restore one validation-selected checkpoint per model.

In [ ]:
# TODO: plot matched learning curves and diagnose differences.

**Locked plain-AE checkpoint and rule:** TODO  
**Locked denoising-AE checkpoint and rule:** TODO

Use the cached test corruptions in the next cells. Do not regenerate easier noise for one model.

In [ ]:
# TODO: report per-pixel test MSE for identity, plain AE, and denoising AE at all four noise levels.
# Plot one robustness curve. Remember that the identity baseline must have zero error at sigma=0.

In [ ]:
# TODO: for all 12 fixed positions, show rows of clean, noisy (sigma=0.35), plain output, and denoising output.
# Do not replace unattractive examples.

In [ ]:
# TODO: over at least 50 held-out images, compare latent distances from clean to own corrupted image
# against distances from clean to unrelated images. Plot distributions for both models and interpret scale cautiously.

### B3. Digits conclusions

1. Why is the noisy image the input but not the target? **TODO**
2. Where does the denoising model improve over the plain model and identity? **TODO**
3. Which kinds of corruption were not tested? **TODO**
4. Can output robustness improve without latent invariance clearly improving? **TODO**

# Lab C — Variational Autoencoder on Fashion-MNIST

Fashion-MNIST is downloaded from the dataset provider through torchvision. This template uses a fixed, stratified CPU-sized subset: 10,000 training, 2,000 validation, and 2,000 official-test examples. Labels establish fixed coverage and are revealed for the final plot only; they must not affect the VAE loss or checkpoint/beta selection.

In [ ]:
fashion_transform = transforms.ToTensor()
fashion_train_full = datasets.FashionMNIST(
    root=DATA_ROOT, train=True, transform=fashion_transform, download=True
)
fashion_test_full = datasets.FashionMNIST(
    root=DATA_ROOT, train=False, transform=fashion_transform, download=True
)

fashion_train_targets = np.asarray(fashion_train_full.targets)
fashion_test_targets = np.asarray(fashion_test_full.targets)

all_fashion_train_idx = np.arange(len(fashion_train_full))
fashion_pool_idx, _ = train_test_split(
    all_fashion_train_idx,
    train_size=12_000,
    random_state=SPLIT_SEED,
    stratify=fashion_train_targets,
)
fashion_train_idx, fashion_val_idx = train_test_split(
    fashion_pool_idx,
    train_size=10_000,
    random_state=SPLIT_SEED,
    stratify=fashion_train_targets[fashion_pool_idx],
)
all_fashion_test_idx = np.arange(len(fashion_test_full))
fashion_test_idx, _ = train_test_split(
    all_fashion_test_idx,
    train_size=2_000,
    random_state=SPLIT_SEED,
    stratify=fashion_test_targets,
)
assert_disjoint(fashion_train_idx, fashion_val_idx)

fashion_train_loader = DataLoader(
    Subset(fashion_train_full, fashion_train_idx.tolist()),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
fashion_val_loader = DataLoader(
    Subset(fashion_train_full, fashion_val_idx.tolist()), batch_size=256, shuffle=False
)
fashion_test_loader = DataLoader(
    Subset(fashion_test_full, fashion_test_idx.tolist()), batch_size=256, shuffle=False
)

FASHION_CLASS_NAMES = list(fashion_train_full.classes)
FASHION_INTERPOLATION_POSITIONS = [0, 1]
FASHION_SAMPLE_SEED = 20_260_812

first_images, first_labels = next(iter(fashion_train_loader))
print("split sizes:", len(fashion_train_idx), len(fashion_val_idx), len(fashion_test_idx))
print("batch shape/range:", first_images.shape, (first_images.min().item(), first_images.max().item()))
print("class names:", FASHION_CLASS_NAMES)

### C1. Define a two-dimensional VAE

The encoder must return `mu` and `log_var`. Use reparameterized sampling during training. Define a separate `decode` method for the grid and prior samples.

In [ ]:
class FashionVAE(nn.Module):
    def __init__(self, latent_dim: int = 2) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: define a CPU-friendly encoder trunk, separate mu/log_var heads, and decoder.

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO
        raise NotImplementedError

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        # TODO: implement z = mu + std * epsilon without breaking the gradient path.
        raise NotImplementedError

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO: return (N, 1, 28, 28) in [0, 1].
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

In [ ]:
def vae_loss(
    reconstruction: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    beta: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # TODO: calculate reconstruction and KL terms with a stated, consistent normalization.
    # Return (total, reconstruction_term, kl_term).
    raise NotImplementedError

# TODO: instantiate a VAE and verify every shape.
# Run one backward pass and prove that both mu-head and log-var-head parameters receive finite gradients.
# Verify numerically that KL is zero when mu=0 and log_var=0.

### C2. Beta comparison plan

Choose the alternative beta before training: **TODO: 0.25 or 4**

| Run | Beta | Architecture | Optimizer | Learning rate | Batch size | Max epochs | Checkpoint rule |
|---|---:|---|---|---:|---:|---:|---|
| Reference | 1 | TODO | TODO | TODO | 128 | TODO | TODO |
| Alternative | TODO | same | same | same | 128 | same | same |

State the reduction used for reconstruction and KL. Predict how the alternative beta should affect reconstruction, KL, code occupancy, and prior samples.

In [ ]:
# TODO: train beta=1 and the preselected alternative with matched seeds and settings.
# Record train/validation total, reconstruction, and KL terms separately.
# Restore one validation-selected checkpoint per beta and record all metadata.

In [ ]:
# TODO: plot total, reconstruction, and KL learning curves separately for both beta runs.
# Do not hide a component merely because its numeric scale is smaller.

**Locked beta=1 checkpoint and rule:** TODO  
**Locked alternative-beta checkpoint and rule:** TODO

The next cells evaluate the official-test subset. Do not revise beta or checkpoint choices after seeing them.

In [ ]:
# TODO: report test reconstruction and KL terms separately for both locked models.
# Collect test mu values. Plot each latent space once without labels and once colored by the ten labels.
# State why mu, rather than a stochastic z sample, is used for this plot.

In [ ]:
# TODO: decode the same fixed 2D coordinate grid for both beta models.
# State the coordinate range and compare it with the distribution of encoded test mu values.
# Explain why the corners of a rectangular [-3, 3] grid are not equally likely under Normal(0, I).

In [ ]:
# TODO: encode the two fixed official-test positions, interpolate with 11 evenly spaced points,
# and decode the path with both beta models. Display all endpoints and intermediate reconstructions.

In [ ]:
# TODO: using FASHION_SAMPLE_SEED, draw the same 64 two-dimensional codes from Normal(0, I).
# Decode and display every sample for both beta models. Do not select favorites.

### C3. VAE conclusions

1. Why does the encoder return `mu` and `log_var`? **TODO**
2. Why does reparameterization preserve a useful gradient path? **TODO**
3. What did the KL term encourage, and what changed at the alternative beta? **TODO**
4. Why do good reconstructions not guarantee good prior samples? **TODO**
5. Which beta evidence is numerical, and which remains a cautious visual judgment? **TODO**

# 5. Latent-Space Field Guide

Complete one row for every trained representation. Do not compare raw losses across datasets.

| Dataset | Model | Code dimension | Constraint | Reconstruction evidence | Neighborhood or robustness evidence | Sampling evidence | Main limitation |
|---|---|---:|---|---|---|---|---|
| Wine | PCA | 2 | Linear orthogonal projection | TODO | TODO | Not a generative prior | TODO |
| Wine | Undercomplete AE | 2 | Bottleneck | TODO | TODO | TODO | TODO |
| Digits | Plain conv AE | TODO | Bottleneck | TODO | TODO | TODO | TODO |
| Digits | Denoising conv AE | TODO | Bottleneck + corruption | TODO | TODO | TODO | TODO |
| Fashion-MNIST | VAE beta=1 | 2 | KL to standard normal | TODO | TODO | TODO | TODO |
| Fashion-MNIST | VAE beta=TODO | 2 | Beta-weighted KL | TODO | TODO | TODO | TODO |

## Synthesis (250–400 words)

1. How did the bottleneck, corruption objective, and KL regularization create different latent spaces?
2. Which model best supports interpolation? Which best supports sampling from a known prior? What evidence supports each?
3. Name three conclusions that the plots do not establish.
4. Give one example of a useful latent space that would not form clean class clusters.

**TODO**

# 6. Critical AI Review and Reproducibility

Ask Codex to inspect labels, preprocessing, range/loss contracts, VAE loss, test-set tuning, cherry-picking, and unsupported claims.

**Suggestion accepted:** TODO  
**Evidence and resulting change:** TODO

**Suggestion changed or rejected:** TODO  
**Evidence and decision:** TODO

**Restart and run-all result:** TODO  
**Any warnings or resource-driven deviations:** TODO  
**Final test evaluation was run only after choices were locked:** `<yes/no; explain>`